In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, accuracy_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.feature_selection import RFE
from sklearn.model_selection import cross_val_score

In [4]:
df = pd.read_csv('student_lifestyle_performance_dataset.csv')

In [6]:
df.head(10)

,Age,Branch,Study_Hours_per_Day,Sleep_Hours,Screen_Time_Hours,Gym_Hours_per_Week,Diet_Type,Attendance_Percentage,Stress_Level_1_to_10,Residence,Internal_Marks,CGPA
0,23,ECE,4.14,6.84,9.23,2.67,Non-Veg,81.24,4.93,Hosteller,65.86,7.52
1,20,Civil,5.97,5.52,3.09,15.61,Veg,90.55,6.96,Day Scholar,62.52,7.21
2,24,Electrical,3.19,3.39,5.02,2.52,Veg,69.40,7.38,Hosteller,40.11,4.84
3,21,CSE,4.77,6.44,9.21,0.00,Non-Veg,80.79,5.84,Day Scholar,61.25,6.74
4,23,Civil,5.42,6.54,4.76,9.93,Veg,82.63,6.67,Day Scholar,64.54,7.77
5,19,ECE,3.01,7.25,5.07,13.89,Non-Veg,80.43,3.28,Hosteller,64.40,7.26
6,23,ECE,2.77,6.24,1.66,8.00,Non-Veg,76.35,3.30,Day Scholar,57.73,6.87
7,24,CSE,3.77,4.71,3.94,9.44,Veg,76.02,4.66,Hosteller,60.32,7.04
8,21,Mechanical,3.31,5.51,6.11,17.94,Veg,79.34,5.80,Day Scholar,54.83,5.90
9,20,IT,5.73,8.08,9.86,13.97,Non-Veg,87.37,6.06,Day Scholar,72.57,7.70


In [ ]:
df.columns

Index(['Age', 'Branch', 'Study_Hours_per_Day', 'Sleep_Hours',
       'Screen_Time_Hours', 'Gym_Hours_per_Week', 'Diet_Type',
       'Attendance_Percentage', 'Stress_Level_1_to_10', 'Residence',
       'Internal_Marks', 'CGPA'],
      dtype='str')

In [24]:
df["pass"] = df["Internal_Marks"].apply(lambda x: 1 if x >= 60.0 else 0)


In [25]:
df.sample(10)

,Age,Branch,Study_Hours_per_Day,Sleep_Hours,Screen_Time_Hours,Gym_Hours_per_Week,Diet_Type,Attendance_Percentage,Stress_Level_1_to_10,Residence,Internal_Marks,CGPA,pass
150,23,ECE,1.76,5.26,1.35,4.37,Non-Veg,61.72,3.29,Day Scholar,50.36,6.11,0
808,25,ECE,6.59,8.14,6.44,16.02,Non-Veg,94.92,5.13,Hosteller,91.75,8.98,1
592,23,Civil,3.72,5.69,2.98,14.42,Veg,77.90,5.74,Day Scholar,54.48,5.65,0
231,24,Civil,3.82,8.35,2.65,8.61,Non-Veg,77.41,2.35,Hosteller,74.45,7.85,1
127,23,Civil,3.67,7.04,4.64,4.48,Veg,81.96,3.88,Day Scholar,59.91,7.58,0
412,23,Civil,3.65,6.68,3.01,2.53,Veg,81.22,2.47,Day Scholar,66.81,7.71,1
9,20,IT,5.73,8.08,9.86,13.97,Non-Veg,87.37,6.06,Day Scholar,72.57,7.70,1
18,18,Civil,4.26,7.17,3.56,5.26,Veg,80.83,3.98,Day Scholar,71.30,8.78,1
98,19,Civil,2.26,4.85,8.32,4.33,Non-Veg,67.36,5.19,Day Scholar,50.79,6.18,0
726,24,Electrical,5.59,7.31,2.99,4.84,Veg,86.87,4.03,Day Scholar,78.59,8.64,1


In [26]:
df["pass"].value_counts()

pass
1    718
0    282
Name: count, dtype: int64

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    1000 non-null   int64  
 1   Branch                 1000 non-null   str    
 2   Study_Hours_per_Day    1000 non-null   float64
 3   Sleep_Hours            1000 non-null   float64
 4   Screen_Time_Hours      1000 non-null   float64
 5   Gym_Hours_per_Week     1000 non-null   float64
 6   Diet_Type              1000 non-null   str    
 7   Attendance_Percentage  1000 non-null   float64
 8   Stress_Level_1_to_10   1000 non-null   float64
 9   Residence              1000 non-null   str    
 10  Internal_Marks         1000 non-null   float64
 11  CGPA                   1000 non-null   float64
 12  pass                   1000 non-null   int64  
dtypes: float64(8), int64(2), str(3)
memory usage: 101.7 KB


In [30]:
cat_cols = df.select_dtypes(include=['str']).columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

In [31]:
df[num_cols].groupby("pass").mean()

,Age,Study_Hours_per_Day,Sleep_Hours,Screen_Time_Hours,Gym_Hours_per_Week,Attendance_Percentage,Stress_Level_1_to_10,Internal_Marks,CGPA
pass,,,,,,,,,
0,20.836879,3.261418,5.811099,4.836418,7.097801,76.091844,4.845426,53.898688,6.353511
1,20.922006,4.342967,6.805432,4.997270,7.396003,81.652423,4.479721,70.644777,7.698621


In [37]:
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
X = df_encoded.drop(columns=["pass", "Internal_Marks"])
y = df_encoded["pass"]
df_encoded.head()

,Age,Study_Hours_per_Day,Sleep_Hours,Screen_Time_Hours,Gym_Hours_per_Week,Attendance_Percentage,Stress_Level_1_to_10,Internal_Marks,CGPA,pass,Branch_Civil,Branch_ECE,Branch_Electrical,Branch_IT,Branch_Mechanical,Diet_Type_Veg,Residence_Hosteller
0,23,4.14,6.84,9.23,2.67,81.24,4.93,65.86,7.52,1,False,True,False,False,False,False,True
1,20,5.97,5.52,3.09,15.61,90.55,6.96,62.52,7.21,1,True,False,False,False,False,True,False
2,24,3.19,3.39,5.02,2.52,69.40,7.38,40.11,4.84,0,False,False,True,False,False,True,True
3,21,4.77,6.44,9.21,0.00,80.79,5.84,61.25,6.74,1,False,False,False,False,False,False,False
4,23,5.42,6.54,4.76,9.93,82.63,6.67,64.54,7.77,1,True,False,False,False,False,True,False


In [ ]:
df_encoded